In [1]:
### notebook to check the magnitude of disorder potential element 

In [1]:
import sys
import os

# Add the parent directory to sys.path
notebook_path = os.getcwd()  # Get the current working directory where the notebook is located
parent_directory = os.path.dirname(notebook_path)  # Move to the parent directory
sys.path.append(parent_directory)

# Now you can import config
from config import *
import Minimal_Model_2D as mm2D
import time

In [2]:

NSM = 60           # number
M = 60             # Note: M*Ay should atleast be 3 times LY
AX = 5             # lattice spacing in nm
AY = 5             # lattice spacing in nm
DELTA = 0.165      # in meV
MEFF_SC = 0.023    # proximitized InAs
MEFF_SM = 0.023    # InAs
NSC = 600
NTOT = 2*NSC + NSM
PHI = 0.5*np.pi 
ALPHAT = 400  #meV-nm^3 # 20 is within perturbative regime for widths atleast up to 100 nm
K_NUM = 4          # No of eigenvalues needed to be found 
#W_MIN = 50

LY = 100           # Width in nm
OMEGA = 2*4*hbm/LY**2/MEFF_SC

MU_SC =  0.8 + OMEGA/2   #in between two subbands
MU_SM =  0.8 + OMEGA/2   #in between two subbands



In [27]:
#### First find wavefunction of SNS with cubic spin orbit coupling

t1 = time.time()

print(t1)

Fcu = mm2D.twoD_Minimal_Model(NTOT, NSM, NSC, M, AX, AY, OMEGA, ALPHAT, DELTA, MEFF_SC, MEFF_SM, MU_SC, MU_SM)
            
#Hcu = Fcu.H_sys(PHI, so = 'cu', uncorr_var=1, lmbda=1)
Hcu = Fcu.H_sys(PHI, so = 'cu')
Ecu, V = eigsh(Hcu ,k=K_NUM ,sigma=0, return_eigenvectors=True)

# Get the indices that would sort Ecu
sorted_indices = np.argsort(Ecu)

# Sort Ecu using the sorted indices
sorted_Ecu = Ecu[sorted_indices]

# Sort the columns of V according to the sorted indices
sorted_V = V[:, sorted_indices]

print(sorted_Ecu, sorted_V.shape)

WV_ref = sorted_V[:, K_NUM//2]

WV_ex = sorted_V[:, K_NUM//2+1]

# Save the array
np.save('GS_wv_SNS_withso.npy', WV_ref)

np.save('FE_wv_SNS_withso.npy', WV_ex)

print(np.sum(np.abs(WV_ref)**2))

t2 = time.time()

print(t2-t1)

1717786982.3708189
[-0.07573829 -0.07544081  0.07544081  0.07573829] (302400, 4)
1.0000000000000022
254.55496382713318


In [8]:
pwd

'c:\\Users\\Vivrekar\\Desktop\\Spin_Splitting_Ge_ABS\\disorder_effect_on_spin_Splitting'

In [29]:
# Load the array
WV_ref = np.load("C:\\Users\\Vivrekar\\Desktop\\Spin_Splitting_Ge_ABS\\disorder_effect_on_spin_Splitting\\GS_wv_SNS_withso.npy")

WV_ex = np.load("C:\\Users\\Vivrekar\\Desktop\\Spin_Splitting_Ge_ABS\\disorder_effect_on_spin_Splitting\\FE_wv_SNS_withso.npy")

In [5]:
#### First find wavefunction of SNS with cubic spin orbit coupling

t1 = time.time()

print(t1)

Fcu = mm2D.twoD_Minimal_Model(NTOT, NSM, NSC, M, AX, AY, OMEGA, ALPHAT, DELTA, MEFF_SC, MEFF_SM, MU_SC, MU_SM)

UNCORR_VAR = 1167

LMBDA = 1e-5

V = Fcu.Disorder_full(UNCORR_VAR,LMBDA)

t2 = time.time()

print(t2-t1)

1717780328.4634223
0.004004716873168945


In [6]:
print(V.shape)

(75600, 75600)


In [9]:
print(V.toarray())

[[ 9.60000183e-04  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00 -4.79929144e-05  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  1.12239552e-04 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 ...
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ... -3.18989313e-04
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
  -1.32293811e-05  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00 -3.04040187e-04]]


In [10]:
WV_ref.shape

(302400,)

In [12]:
Va = V.toarray()

In [13]:
Va.shape

(75600, 75600)

To Find overlap integral of disorder potential with gs wv i.e. 

$\langle \Psi_0| V_{dis} | \Psi_0 \rangle $ where $\Psi = (\psi_{e\uparrow},\psi_{e\downarrow},\psi_{h\uparrow},\psi_{h\downarrow})^T $ 

We get: $\langle \Psi_0| V_{dis} | \Psi_0 \rangle  = \sum \int dr V(r)|(\psi_{e\uparrow}(r)|^2 + |\psi_{e\downarrow}(r)|^2 - |\psi_{h\uparrow}(r)|^2 - |\psi_{h\downarrow}(r)|^2$)

In [14]:
WV_ref_reshaped = WV_ref.reshape(2,2,M*NTOT)

In [17]:
S = np.abs(WV_ref_reshaped)**2

In [18]:
V.shape, WV_ref_reshaped.shape, S.shape

((75600, 75600), (2, 2, 75600), (2, 2, 75600))

In [22]:
VOI = np.empty((2,2))

for i in range(2):

    for j in range(2):

        VOI[i,j] = np.sum(V@S[i,j,:])



In [23]:
print(VOI)

[[1.53930181e-07 1.59808321e-07]
 [2.00289512e-07 1.97280517e-07]]


In [26]:
(VOI[0,0] + VOI[0,1] - VOI[1,0] - VOI[1,1])/DELTA

-5.080698625332679e-07

To Find overlap integral of disorder potential with gs wv and fe wv i.e. 

$\langle \Psi_1| V_{dis} | \Psi_0 \rangle $ where $\Psi = (\psi_{e\uparrow},\psi_{e\downarrow},\psi_{h\uparrow},\psi_{h\downarrow})^T $ 

We get: $\langle \Psi_1| V_{dis} | \Psi_0 \rangle  = \sum_{i=(e\uparrow, e\downarrow, h\uparrow, h\downarrow)} \int dr V(r)\psi_{e1i}^{*}(r)\psi_{e0i}(r) $

In [ ]:
WV_ex_reshaped = WV_ex.reshape(2,2,M*NTOT)

#S = np.abs(WV_ex_reshaped)**2

V.shape, WV_ex_reshaped.shape, S.shape

VOI_ge = np.empty((2,2))

for i in range(2):

    for j in range(2):

        VOI_ge[i,j] = np.sum(V@S[i,j,:])
